In [1]:
import re
# import emoji
import warnings
warnings.filterwarnings('ignore')

# import spacy
import pandas as pd
import torch
from transformers import pipeline
from sklearn.cluster import KMeans

from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer

from nltk.corpus import stopwords
stop_words = stopwords.words('russian')

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

from razdel import tokenize

from sklearn.feature_extraction.text import CountVectorizer

from keybert import KeyBERT

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, TextGeneration
from bertopic.vectorizers import ClassTfidfTransformer  
from bertopic.dimensionality import BaseDimensionalityReduction



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vallo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [3]:
news = pd.read_csv("../data/raw/nnews.csv")

In [4]:
data = pd.read_csv("../data/raw/csv-dump.csv")
data.head()

,global_post_id,source_id,entity_id,entity,sentiment_id,genre_id,date,view_counter,repost_counter,title,text,link
0,2_19761,2,107,РЖД,2,1,2025-04-18 6:32:09,13494,6,Главное к открытию пятницы (18.04): 👉 Значение...,Главное к открытию пятницы (18.04):\n\n👉 Значе...,https://t.me/AK47pfl/19761
1,2_19761,2,93,ЛСР,2,2,2025-04-18 6:32:09,13494,6,Главное к открытию пятницы (18.04): 👉 Значение...,Главное к открытию пятницы (18.04):\n\n👉 Значе...,https://t.me/AK47pfl/19761
2,6_13638,6,54,SOKOLOV,3,7,2025-04-19 8:04:24,4308,35,"За полгода АвтоВАЗ продал один ""суверенный эле...","За полгода АвтоВАЗ продал один ""суверенный эле...",https://t.me/Alekhin_Telega/13638
3,6_13638,6,82,Lada,3,7,2025-04-19 8:04:24,4308,35,"За полгода АвтоВАЗ продал один ""суверенный эле...","За полгода АвтоВАЗ продал один ""суверенный эле...",https://t.me/Alekhin_Telega/13638
4,7_2715,7,314,Альфа-Банк,1,2,2025-04-18 11:00:37,312502,402,Официально: мы вложили деньги в банку. В Альфа...,Официально: мы вложили деньги в банку. В Альфа...,https://t.me/AlfaBank/2715


In [5]:
data.shape

(898, 12)

In [6]:
data.isna().sum()


global_post_id    0
source_id         0
entity_id         0
entity            0
sentiment_id      0
genre_id          0
date              0
view_counter      0
repost_counter    0
title             0
text              0
link              0
dtype: int64

In [12]:
def clean_text(text):

    # Приведение текста к нижнему регистру
    # text = text.lower()s
    # text = emoji.replace_emoji(text, '')

    # Замена всех не-словесных символов на пробел (кроме букв и знаков препинания)
    text = re.sub(r'\W+', ' ', text)

    # Удаление URL-адресов
    text = re.sub(r"http\S+", "", text)

    # Создание шаблона для HTML-тегов
    html = re.compile(r'&lt;.*?&gt;')

    # Удаление HTML-тегов из текста
    text = html.sub(r'', text)

    # Список пунктуаций для удаления
    punctuations = '@#!?+&amp;*[]-%.:/();$=&gt;&lt;|{}^' + "'`" + '_'
    for p in punctuations:
        text = text.replace(p, '')  # Удаление пунктуации

    # Удаление стоп-слов и приведение слов к нижнему регистру
    text = [word for word in text.split() if word.lower() not in stop_words]

    # Объединение слов обратно в текст
    text = " ".join(text)

    # Создание шаблона для поиска эмодзи
    emoji_pattern = re.compile("["
                        u"\U0001F600-\U0001F64F"  # эмоции
                        u"\U0001F300-\U0001F5FF"  # символы и пиктограммы
                        u"\U0001F680-\U0001F6FF"  # транспорт и карты
                        u"\U0001F1E0-\U0001F1FF"  # флаги
                        u"\U00002702-\U000027B0"
                        u"\U000024C2-\U0001F251"
                        "]+", flags=re.UNICODE)

    # Удаление эмодзи из текста
    text = emoji_pattern.sub(r'', text)

    return text


In [13]:
# Создаем новый столбец с очищенным текстом
data['cleaned_text'] = data['text'].apply(clean_text)

In [14]:
data['cleaned_text'].iloc[0]

'Главное открытию пятницы 18 04 Значение индекса ЖиС 53 спокойствие Подробнее индексе 11 06 67 85 барр Трамп рассчитывает Вашингтон Пекин смогут выйти договоренности торговле течение ближайших трех четырех недель Трамп сообщил ближайшее время ожидает реакции Москвы инициативу прекращении огня Украине Макрон анонсировал следующий этап обсуждений урегулирования конфликта Украине пройдут Лондоне следующей неделе Россия успешно минимизировала использование недружественных валют Доля доллара евро сократилась международных резервах платежах РЖД вернулась идее запуска rde in грузовых вагонов События сегодня 1 Отсечка ЦМТ WTCM 2 ГОСА дивидендам 2024 ЛСР LSRG 3 Торги США Гонконге Германии других странах Европы проводятся связи пасхальными праздниками влияет рынки ближайшие 5 дней Доступно членам RDVPREMIUMbo Аналитика by AK47f'

In [10]:
news.dropna(subset=['text'], inplace=True)

def clean_date(date_str):
    try:
        # Используем регулярное выражение для извлечения даты в формате YYYY-MM-DD
        match = re.search(r'(\d{4}-\d{2}-\d{2})', str(date_str))
        if match:
            return match.group(1)
        return date_str
    except:
        return date_str

news['date'] = news['date'].apply(clean_date)
news.shape

(1832, 4)

# Text classification

In [15]:
embedding_model = SentenceTransformer("deepvk/USER2-base", device=device)


# Topic modeling

In [2]:
# embedding_model_name = "deepvk-USER2-base"
embedding_model_name = "cointegrated/rubert-tiny"
representation_model_name = 'cointegrated/rut5-small'

In [34]:
def tokenize_ru(text):
    words = tokenize(text)
    return [word.text for word in words]

In [35]:
dim_model = UMAP(n_neighbors=15, n_components=50, min_dist=0.0, metric='euclidean')
cluster_model = HDBSCAN(min_cluster_size=5, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
# cluster_model = KMeans(n_clusters=10, random_state=42)
vectorizer_model = CountVectorizer(tokenizer=tokenize_ru, ngram_range=(1, 2), stop_words=stop_words)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [40]:

representation_model = KeyBERTInspired()

generator = pipeline('text2text-generation', model=representation_model_name, device=device, batch_size=100)
representation_model = TextGeneration(generator)

Device set to use cpu


In [43]:
topic_model = BERTopic(
  language="russian",
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=dim_model,                     # Step 2 - Reduce dimensionality
  hdbscan_model=cluster_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)

In [44]:
topics, probs = topic_model.fit_transform(data['cleaned_text'])

In [45]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,142,"-1_Название of this topic is, где, rc, lc, ___","[Название of this topic is, где, rc, lc, , , ,...",[планировал поехать США выбрал программу серти...
1,0,39,"0_Название of this topic is, где недвижимости,...","[Название of this topic is, где недвижимости, ...",[ЧАСТЬ 4 ВСЁ СМЕШАЛОСЬ ЛЮДИ КОНИ 2019 году сме...
2,1,36,"1_Название of this topic is, когда, разборы, к...","[Название of this topic is, когда, разборы, ко...",[Разборы компаний Павла Шумилова Приветствую п...
3,2,31,"2_Название of this topic is, выше, потенциал, ...","[Название of this topic is, выше, потенциал, м...",[Рынок акций утром пятницу открылся снижением ...
4,3,29,"3_Название of this topic is, где, сверления, м...","[Название of this topic is, где, сверления, мо...",[Вечерняя подборка товаров выгодным ценам AiEx...
5,4,26,"4_, oon, brin, brin, vosok, vosok,___","[, oon, brin, brin, vosok, vosok,, , , , , , ,...",[Ozon заходит российский футбол честь назвали ...
6,5,25,"5_Что означает, когда, гостеприимства, арендод...","[Что означает, когда, гостеприимства, арендода...",[апреле Авито Путешествиям исполняется год быс...
7,6,24,"6_Название of this topic, где aceyed, aceyed___","[Название of this topic, где aceyed, aceyed, ,...",[Стереть ноги это мозоль пятке новой обуви неп...
8,7,22,"7_Название of this topic is, когда, когда моск...","[Название of this topic is, когда, когда москв...",[Лови большую пачку кодов крутыми горящими пре...
9,8,21,"8_Название of this topic, сахарной, куличи, по...","[Название of this topic, сахарной, куличи, пом...",[Продолжаем тур куличам Яндекс Еда рестораны п...


In [270]:
# Скрываем предупреждения

topics, probs = topic_model.fit_transform(news['text'])

In [271]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,353,0_бизнес_forbes_эксперты_рассказываем,"[бизнес, forbes, эксперты, рассказываем, техно...","[Начал вещание первый сервис, позволяющий росс..."
1,1,280,1_сообщалось_ребенка_обнаружили_года,"[сообщалось, ребенка, обнаружили, года, ребено...",[Россияне на всю жизнь оставили своего ребенка...
2,2,231,2_россиян_законопроект_закон_службе,"[россиян, законопроект, закон, службе, российс...",[Госдума приняла в первом чтении законопроект ...
3,3,177,3_президента_лавровым_американский_россия,"[президента, лавровым, американский, россия, а...",[Президент США Дональд Трамп понимает выгоду о...
4,4,163,4_донецкой_украину_переговорах_переговоров,"[донецкой, украину, переговорах, переговоров, ...","[Президент Украины Владимир Зеленский считает,..."
5,5,152,5_убийству_убийстве_убийства_кузьменко,"[убийству, убийстве, убийства, кузьменко, взры...",[Во время встречи канцлера Германии Ангелы Мер...
6,6,148,6_мосбирже_ставку_мосбиржа_года,"[мосбирже, ставку, мосбиржа, года, 2024, 2023,...",[Несмотря на беспрецедентную жесткость денежно...
7,7,131,7_reuters_инфляции_бирже_турции,"[reuters, инфляции, бирже, турции, bloomberg, ...","[Пошлины на импорт, которые Трамп обещал еще в..."
8,8,125,8_пожар_возгорание_пожара_крейсер,"[пожар, возгорание, пожара, крейсер, кузнецов,...",[Число пострадавших при пожаре на авианесущем ...
9,9,72,9_спорт_олимпийского_олимпийская_олимпийских,"[спорт, олимпийского, олимпийская, олимпийских...",[Глава Федерации лыжных гонок России (ФЛГР) Ел...


In [46]:
ngram_range = "3x3"

In [47]:
# Создаем более осмысленные названия для топиков
# Используем KeyBERT для генерации ключевых фраз из документов каждого топика


# Инициализируем модель KeyBERT
keybert_model = KeyBERT(model=embedding_model)

# Получаем информацию о топиках
topic_info = topic_model.get_topic_info()
topic_docs = {}

# Для каждого топика (кроме -1, который означает выбросы) получаем репрезентативные документы
for topic_id in topic_info[topic_info['Topic'] != -1]['Topic']:
    # Получаем документы для данного топика
    documents = topic_model.get_representative_docs(topic_id)
    topic_docs[topic_id] = ' '.join(documents)

# Создаем словарь для хранения новых названий топиков
topic_names = {}

# Для каждого топика генерируем ключевые фразы
for topic_id, doc in topic_docs.items():
    # Извлекаем ключевые фразы (2 слова) из документов топика
    keywords = keybert_model.extract_keywords(doc, keyphrase_ngram_range=(3, 3), stop_words=stop_words, top_n=1)
    
    if keywords:
        # Берем первую ключевую фразу как название топика
        topic_names[topic_id] = keywords[0][0]
    else:
        # Если не удалось извлечь фразу, используем оригинальное название
        words = topic_model.get_topic(topic_id)
        topic_names[topic_id] = f"Топик_{topic_id}_{words[0][0]}_{words[1][0]}"

# Переименовываем топики в модели
topic_model.set_topic_labels(topic_names)

# Выводим обновленную информацию о топиках
# print("Топики с новыми названиями:")
# display(topic_model.get_topic_info()[1:11])


In [48]:
topic_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,-1,142,"-1_Название of this topic is, где, rc, lc, ___","-1_Название of this topic is, где, rc, lc, ___","[Название of this topic is, где, rc, lc, , , ,...",[планировал поехать США выбрал программу серти...
1,0,39,"0_Название of this topic is, где недвижимости,...",проект ипотеку раздавали,"[Название of this topic is, где недвижимости, ...",[ЧАСТЬ 4 ВСЁ СМЕШАЛОСЬ ЛЮДИ КОНИ 2019 году сме...
2,1,36,"1_Название of this topic is, когда, разборы, к...",прогнозирование акциями дальше,"[Название of this topic is, когда, разборы, ко...",[Разборы компаний Павла Шумилова Приветствую п...
3,2,31,"2_Название of this topic is, выше, потенциал, ...",ждут акции золотодобывающих,"[Название of this topic is, выше, потенциал, м...",[Рынок акций утром пятницу открылся снижением ...
4,3,29,"3_Название of this topic is, где, сверления, м...",яндекс маркет беспроводные,"[Название of this topic is, где, сверления, мо...",[Вечерняя подборка товаров выгодным ценам AiEx...
5,4,26,"4_, oon, brin, brin, vosok, vosok,___",ozon ареной также,"[, oon, brin, brin, vosok, vosok,, , , , , , ,...",[Ozon заходит российский футбол честь назвали ...
6,5,25,"5_Что означает, когда, гостеприимства, арендод...",бизнес авито путешествиями,"[Что означает, когда, гостеприимства, арендода...",[апреле Авито Путешествиям исполняется год быс...
7,6,24,"6_Название of this topic, где aceyed, aceyed___",гиалуроновой кислоты интересненькие,"[Название of this topic, где aceyed, aceyed, ,...",[Стереть ноги это мозоль пятке новой обуви неп...
8,7,22,"7_Название of this topic is, когда, когда моск...",промокод gedjt8pj скидка,"[Название of this topic is, когда, когда москв...",[Лови большую пачку кодов крутыми горящими пре...
9,8,21,"8_Название of this topic, сахарной, куличи, по...",петербурге шоколадный кулич,"[Название of this topic, сахарной, куличи, пом...",[Продолжаем тур куличам Яндекс Еда рестораны п...


In [274]:
topic_model.get_topic_info()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,0,353,0_бизнес_forbes_эксперты_рассказываем,forbes_education forbes_young forbesfranchises,"[бизнес, forbes, эксперты, рассказываем, техно...","[Начал вещание первый сервис, позволяющий росс..."
1,1,280,1_сообщалось_ребенка_обнаружили_года,издевалась детьми хоккеист,"[сообщалось, ребенка, обнаружили, года, ребено...",[Россияне на всю жизнь оставили своего ребенка...
2,2,231,2_россиян_законопроект_закон_службе,повысился налог добавленную,"[россиян, законопроект, закон, службе, российс...",[Госдума приняла в первом чтении законопроект ...
3,3,177,3_президента_лавровым_американский_россия,лаврова президентом сша,"[президента, лавровым, американский, россия, а...",[Президент США Дональд Трамп понимает выгоду о...
4,4,163,4_донецкой_украину_переговорах_переговоров,переговоров нормандской четверки,"[донецкой, украину, переговорах, переговоров, ...","[Президент Украины Владимир Зеленский считает,..."
5,5,152,5_убийству_убийстве_убийства_кузьменко,подозреваемых убийстве шеремета,"[убийству, убийстве, убийства, кузьменко, взры...",[Во время встречи канцлера Германии Ангелы Мер...
6,6,148,6_мосбирже_ставку_мосбиржа_года,российский банковский сектор,"[мосбирже, ставку, мосбиржа, года, 2024, 2023,...",[Несмотря на беспрецедентную жесткость денежно...
7,7,131,7_reuters_инфляции_бирже_турции,хотели отмены пошлин,"[reuters, инфляции, бирже, турции, bloomberg, ...","[Пошлины на импорт, которые Трамп обещал еще в..."
8,8,125,8_пожар_возгорание_пожара_крейсер,пострадавших возгорании крейсере,"[пожар, возгорание, пожара, крейсер, кузнецов,...",[Число пострадавших при пожаре на авианесущем ...
9,9,72,9_спорт_олимпийского_олимпийская_олимпийских,претензий российскому олимпийскому,"[спорт, олимпийского, олимпийская, олимпийских...",[Глава Федерации лыжных гонок России (ФЛГР) Ел...


In [25]:
topic_model.get_topic_info().to_csv(f'../data/interim/dumptopics-{representation_model_name}-{embedding_model_name}-{ngram_range}-HDBScan5.csv', index=False)

In [26]:
topic_model.visualize_topics()

In [23]:
pdd = topic_model.get_topic_info()

In [29]:
pdd["CustomName"].tolist()

['-1_года_разобраться_благами_какие выплаты',
 'проект ипотеку раздавали',
 'прогнозирование акциями дальше',
 'улицы москвы среднем',
 'ждут акции золотодобывающих',
 'яндекс маркет беспроводные',
 'гиалуроновой кислоты интересненькие',
 'ozon ареной также',
 'бизнес авито путешествиями',
 'пост работе контентом',
 'бот называется подарки',
 'люблю патчи aexskin',
 'петербурге шоколадный кулич',
 'концертов 19 00',
 'яндекс маркете доставка',
 'сказать дожили макияж',
 'купил технику хайер',
 'промокод gedjt8pj скидка',
 'продажи самолетов дрлоиу',
 'хитрых фишек ios',
 'оформляем дебетовую карту',
 'boxberry яндекс объявил',
 'массажер пистолет устройство',
 'акне пигментацией кожа',
 'заработать миллион долларов',
 'дом рф застройщик',
 'клиентов заблокированными счетами',
 'рейтинг основателя dns',
 'аренды электросамокатов 2024',
 'ооо вайлдберриз получило',
 'бренда рф ikea',
 'невероятных женщин космос',
 'страхования напоминаем ярдрей',
 'рынок умных домов',
 'бизнес конференци

In [34]:
pdd["CustomName"].tolist()

['-1_года_разобраться_благами_какие выплаты',
 'проект ипотеку раздавали',
 'прогнозирование акциями дальше',
 'улицы москвы среднем',
 'ждут акции золотодобывающих',
 'яндекс маркет беспроводные',
 'гиалуроновой кислоты интересненькие',
 'ozon ареной также',
 'бизнес авито путешествиями',
 'пост работе контентом',
 'бот называется подарки',
 'люблю патчи aexskin',
 'петербурге шоколадный кулич',
 'концертов 19 00',
 'яндекс маркете доставка',
 'сказать дожили макияж',
 'купил технику хайер',
 'промокод gedjt8pj скидка',
 'продажи самолетов дрлоиу',
 'хитрых фишек ios',
 'оформляем дебетовую карту',
 'boxberry яндекс объявил',
 'массажер пистолет устройство',
 'акне пигментацией кожа',
 'заработать миллион долларов',
 'дом рф застройщик',
 'клиентов заблокированными счетами',
 'рейтинг основателя dns',
 'аренды электросамокатов 2024',
 'ооо вайлдберриз получило',
 'бренда рф ikea',
 'невероятных женщин космос',
 'страхования напоминаем ярдрей',
 'рынок умных домов',
 'бизнес конференци

In [35]:
topic_model.visualize_barchart(top_n_topics=30, n_words=5, title='Топ слов по темам', width=400, height=250, custom_labels=True)

## Pipeline with translation

In [27]:
# Создаем датафрейм для работы с моделью
df = news[['text']].copy()
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-ru-en", device=device)

# Функция для перевода текста с русского на английский
def translate_text(text: list[str]) -> list[str]:
        
    
    # Ограничиваем длину текста для перевода (модель имеет ограничения)


    max_length = 512
    if len(text) > max_length:
        text = text[:max_length]
    # Выполняем перевод
    result = translator(" ".join(text))
    
    # Возвращаем переведенный текст
    return result[0]['translation_text'].split(" ")

# Применяем функцию перевода к представлениям тем

translated_representations = []

for topic in topic_representations:
    # Получаем текущее представление темы
    # Переводим слова темы
    translated_words = translate_text(topic)
    # Обновляем название темы с переводом
    translated_representations.append(translated_words)


Device set to use cuda
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


---

## Отрисовка кластеров

In [65]:
# Получаем эмбеддинги документов
embeddings = embedding_model.encode(data['cleaned_text'].tolist(), show_progress_bar=True)
embeddings.shape  # Выводим размерность полученных эмбеддингов

dim_2d_model = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine')
zipped_embs = dim_2d_model.fit_transform(embeddings)
cluster_model.fit(zipped_embs)


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

HDBSCAN(min_cluster_size=10, prediction_data=True)

In [66]:
zipped_embs.shape

(898, 2)

In [168]:
import pandas as pd

data = pd.read_csv('../data/interim/topics-KeyBERTInspired-deepvk-USER2-base-5x5.csv')

In [171]:
data.head()

,Topic,Count,Name,CustomName,Representation,Representative_Docs
0,-1,404,-1_кузьменко_журналиста_антоненко_подозреваемых,-1_кузьменко_журналиста_антоненко_подозреваемых,"['кузьменко', 'журналиста', 'антоненко', 'подо...",['Материалы по делу об убийстве журналиста Пав...
1,0,284,0_сообщалось_обнаружили_года_ребенка,видео посвященное годовщине свадьбы кеосаяном,"['сообщалось', 'обнаружили', 'года', 'ребенка'...","['В Санкт-Петербурге задержали группу парней, ..."
2,1,169,1_reuters_торги_бирже_мосбиржа,лукойл сообщил падении прибыли 27,"['reuters', 'торги', 'бирже', 'мосбиржа', 'инв...",['«Лукойл» сообщил о падении прибыли на 27%: ч...
3,2,124,2_россиян_нацпроекта_российских_получат,которого российских образовательных учреждения...,"['россиян', 'нацпроекта', 'российских', 'получ...",['Росгосцирк предложил Министерству Культуры Р...
4,3,84,3_донецкой_переговорах_переговоров_владимир,проведения встречи нормандском формате украинс...,"['донецкой', 'переговорах', 'переговоров', 'вл...",['На следующей встрече лидеров стран «нормандс...


In [172]:
data['Representation'].tolist()

["['кузьменко', 'журналиста', 'антоненко', 'подозреваемых', 'года', 'левченко', 'шеремета', 'суд', 'российские', 'эксперты']",
 "['сообщалось', 'обнаружили', 'года', 'ребенка', 'инцидент', 'умер', 'ребенок', '2018', 'известно', 'стал']",
 "['reuters', 'торги', 'бирже', 'мосбиржа', 'инвесторам', 'доходность', 'bloomberg', 'акционеры', 'мосбиржи', 'инвесторов']",
 "['россиян', 'нацпроекта', 'российских', 'получат', 'нацпроект', 'региона', 'отметил', 'подмосковье', 'года', 'программы']",
 "['донецкой', 'переговорах', 'переговоров', 'владимир', 'зеленского', 'владимира', 'украина', 'нормандском', 'соглашений', 'положений']",
 "['спорт', 'олимпийского', 'олимпийская', 'олимпийских', 'олимпийские', 'антидопинговое', 'олимпийской', 'спортсменов', 'российское', 'олимпиаде']",
 "['песков', 'чиновников', 'оскорбление', 'закон', 'приговорили', 'госдуму', 'госдума', 'суд', 'депутат', 'суда']",
 "['forbes_young', 'forbes', 'forbes_education', 'рассказываем', 'бизнес', 'бизнесменов', 'дайджест', 'фи

In [75]:
topic_model.topic_embeddings_.shape

(28, 768)

In [73]:
topic_model.visualize_topics()

ImportError: cannot import name '_DelayedCategories' from 'narwhals.dtypes' (c:\Users\R1\Documents\Business\company-clasterization\.venv\Lib\site-packages\narwhals\dtypes.py)

In [67]:
import json

# Проверяем содержимое перед парсингом
print(data['Representative_Docs'].iloc[0])

# Исправляем ошибку парсинга JSON - возможно, строка требует предварительной обработки
try:
    # Пробуем очистить строку от лишних символов и заменить одинарные кавычки на двойные
    cleaned_json = data['Representative_Docs'].iloc[0].replace("'", '"')
    parsed_data = json.loads(cleaned_json)
    print("Успешно распарсили JSON")
    print(parsed_data)
except json.JSONDecodeError as e:
    print(f"Ошибка парсинга JSON: {e}")
    # Альтернативный подход - использовать ast.literal_eval для парсинга Python литералов
    import ast
    try:
        parsed_data = ast.literal_eval(data['Representative_Docs'].iloc[0])
        print("Успешно распарсили с помощью ast.literal_eval")
        print(parsed_data)
    except:
        print("Не удалось распарсить данные")

['Глава МЧС России Евгений Зиничев поддержал назначение своего бывшего заместителя Игоря Кобзева на должность временного исполняющего обязанности губернатора Иркутской области. Об этом сообщает ТАСС. По словам руководителя ведомства, Кобзев в новой должности будет внимательно относиться к проблемам каждого человека и справится с поставленными задачами и выразил надежду, что новый глава региона будет руководствоваться принципами неравнодушного подхода к проблемам каждого жителя Иркутской области. Зиничев рассказал, что проработал с ним полтора года, и за это время Кобзев показал себя как инициативный и грамотный сотрудник. «Под его руководством и при его личном участии была скорректирована модель риск-ориентированного подхода к объектам с массовым пребыванием людей, определены категории риска объектов», — поделился глава МЧС. Назначение также поддержал заместитель председателя правительства России Алексей Гордеев. Он охарактеризовал Кобзева как эффективного и системного руководителя, на

In [23]:
parsed_data

['Глава МВД Украины Арсен Аваков заявил, что Киев и Москва могут прийти к компромиссу в вопросе передачи Украине контроля над границей в Донбассе. Интервью с министром опубликовано на странице издания «Громадське» в Twitter. По его словам, в первое время контроль над границей могут осуществлять не погранвойска, а украинская полиция вместе с «представителями территориальных общин». При этом он указал, что это станет возможным только после того, как вооруженные формирования покинут территории самопровозглашенных республик. Аваков считает, что такой «переходный период» может продолжаться вплоть до года, но Украина «готова это пройти». При этом он указал, что участники «нормандского саммита» не дали согласия на такой вариант, а президент России Владимир Путин «не готов вернуть границу». Говоря о возможном компромиссе, Аваков отметил, что Киев может получить контроль над границей «не за месяц до местных выборов, а за два дня». Ранее президент Украины Владимир Зеленский заявил о необходимост

In [ ]:
topic_model.get_topic_info()["CustomName"]

In [68]:
# Визуализация кластеров новостей в 2D пространстве с помощью plotly
import plotly.express as px
import pandas as pd
import numpy as np

# Получаем координаты документов в 2D пространстве
embeddings_2d = zipped_embs

# Получаем данные о документах
doc_info = topic_model.get_document_info(data['cleaned_text'].tolist())

# Создаем DataFrame для визуализации
plot_df = pd.DataFrame({
    'x': embeddings_2d.embedding_x,
    'y': embeddings_2d.embedding_y,
    'topic': embeddings_2d.topic,
    'text': doc_info.Document,
    'topic_name': doc_info.Name
})

# Создаем цветовую схему
colors = px.colors.qualitative.Plotly

# Создаем интерактивную визуализацию
fig = px.scatter(
    plot_df, 
    x='x', 
    y='y', 
    color='topic_name',
    hover_data=['text'],
    title='Кластеризация новостей по темам',
    color_discrete_sequence=colors,
    opacity=0.7,
    size_max=10
)

# Настраиваем внешний вид графика
fig.update_traces(marker=dict(size=8, line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(
    legend_title_text='Темы',
    xaxis_title="",
    yaxis_title="",
    xaxis=dict(showticklabels=False),
    yaxis=dict(showticklabels=False),
    plot_bgcolor='white'
)

# Отображаем график
fig.show()



AttributeError: 'numpy.ndarray' object has no attribute 'embedding_x'